Data processing in turbulence is critical because it enables the extraction of meaningful insights from complex and chaotic flow data, helping researchers understand turbulent structures, energy transfer, and flow dynamics. Effective data processing ensures accurate analysis, facilitates comparisons with theoretical models, and supports the validation of simulations and experiments. We use Python for this due to its powerful libraries like NumPy, and Pandas for numerical computations and Matplotlib for visualization.

**GOAL:**
By the end of this tutorial, you will have a strong understanding of how to process data using Pandas and NumPy. Additionally, you will be able to visualize the data effectively with Matplotlib, providing you with a solid foundation for analyzing and interpreting turbulent flow data. 

**Prerequisites:**
Although many concepts are explained in detail in this tutorial, a certain level of prior knowledge is still required. If you have never worked with Python before, we recommend taking the [Udacity](https://www.udacity.com/course/introduction-to-python--ud1110) tutorial first.

::: {.callout-tip}
## Why learn this?
A DNS run leaves you with millions of numbers per snapshot — the physics only
appears once you can *slice, reduce and visualise* them. pandas and NumPy are
how you turn that raw table into velocities, statistics and figures without
writing slow Python loops.

- **Imagine you need** the velocity magnitude at every one of eight million
  points — one vectorised line does it, where a `for` loop takes minutes.
- **Imagine you need** to compare the flow field at ten timesteps side by
  side — a subplot loop builds all ten from one block of code.
:::

::: {.callout-note}
## By the end of Part A you will be able to
- load a `.parquet` field into a pandas DataFrame and inspect its structure;
- add, remove and compute new columns (e.g. the velocity magnitude) with
  vectorised NumPy;
- slice the field in space and in time;
- build line plots, filled contours, subplots and density (hexbin) plots; and
- capture a consistent figure style in a Matplotlib stylesheet.
:::

::: {.callout-note}
## Deliverables

This part is a **guided walk-through**: work through it top to bottom, since
each step builds on the previous one. Submit the requested plots and short
answers for the numbered **Tasks** in **Crowdmark** by the posted deadline.
New to the dataset? Start with the
[Assignment 1 overview](assignment-01-dns.qmd).
:::

## Setup

Run this cell once at the top. It imports the libraries used throughout and
applies the shared **course plotting style**, so every figure below is
consistent in font, colours and size without repeating styling code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Shared McGill course style (relative to this tutorial folder)
plt.style.use("../python/plotting-styles/mystyle.mplstyle")

## Handling data with pandas
Pandas is a powerful and widely-used library in Python for data manipulation and analysis. It provides intuitive and efficient tools to handle structured data, such as tables and time series, making it an essential tool for data processing tasks. With pandas, you can load, clean, transform, and analyze data using flexible data structures like DataFrames (2D tabular data) and Series (1D labeled arrays). Whether you're working with simple CSV files or complex datasets, pandas allows you to explore and preprocess your data with ease, enabling seamless integration into machine learning, data visualization, or statistical analysis workflows.

### Loading data into a pandas DataFrame

For this tutorial we use part of a dataset from the
[JHTDB](https://turbulence.idies.jhu.edu/datasets/homogeneousTurbulence/isotropic):
DNS data of a $200^3$ cube of isotropic turbulence, including the first 800
timesteps. For convenience it has been packaged into a single compact
`.parquet` file — see the [Assignment 1 overview](assignment-01-dns.qmd) for
what Parquet is and where the file lives on the cluster.

To get a visual impression of the flow, watch a short
[JHTDB visualization](https://turbulence.idies.jhu.edu/visualizations/sweptThrough).

The first step is to load the file and understand its structure.

In [ ]:
# Load pandas and read the packaged dataset.
# On the cluster the file is at /project/60004/00_Assignment1/isotropic.parquet
import pandas as pd

df = pd.read_parquet("isotropic.parquet")

### Printing the DataFrame
First of all we want to take a first look into the DataFrame. In order to print the DataFrame you can use `print(df_name)`or just type `df_name`. 

In [ ]:
df

In the DataFrame, each point of the mesh in the computational domain is represented by the x,y,z coordinates. The first column “Timestep” indicates the respective time step. This means that a specific point can be assigned to a specific time for each row. For each of these points, the respective velocity components in each spatial direction is stored in further columns.

### Printing the columns names
If we are working with a larger DataFrame, it is practical to first look at the columns only, as we immediately know the parameters that are stored in the Dataframe. Therefore we can use `print(list(df_name))`.

In [ ]:
print(list(df))

### Summary of DataFrame
If you want to get an overview of the DataFrame first you can use the function `info()`.
You can use it and see what information you get out of it.

In [ ]:
df.info()

### Access specific points and values
To access certain columns we can use `df_name['column_name']`.

In [ ]:
df['velocity_x']

With `.iloc[row]` certain rows can be accessed.

In [ ]:
df.iloc[0]

Finally, access a specific data frame entry by selecting the corresponding column and row.

::: {.callout-important}
## Task 1

Print the velocity in the $x$ direction at the **last** point of the DataFrame.

::: {.callout-tip collapse="true"}
## Hint
Negative indexing lets you access elements in reverse order.
:::
:::

#### Adding columns
We can add new columns by giving them a new name and assigning specific values. For example, we could add the column “Temperature” and give the value 300 for each point, i.e. each row.

In [ ]:
#Creates a new column with all the values equal to 300
df['Temperature'] = 300
df

Since a column with only the same entries makes little sense, it is more likely that additional data, which is stored in a list for example, must be added to the DataFrame. 

In [ ]:
# Generates a list with random numbers between 290 and 1000 in the length of the data frame.
import random
temperature = [random.randint(290, 1000) for _ in range(len(df))]

# Adds the list to the DataFrame
df['Temperature'] = temperature
df

Specific entries can also be changed by specifying the corresponding row.

In [ ]:
df.loc[4, 'Temperature'] = 1000
df.loc[4, 'Temperature']

#### Removing columns and rows
To remove the column again we can use the function `drop('column_came', axis=1)`. You can also remove specific rows by using `drop(index, axis=0)`. The 0 for axis indicates that a row is to be removed and 1 for a column.

In [ ]:
# Removing the column Temperature from the DataFrame
df = df.drop('Temperature', axis=1)
df

### Calculating with DataFrames

In order to make our calculations as simple and efficient as possible we use Numpy.

NumPy is a powerful Python library for numerical computing and the foundation for many data science and machine learning tools. It provides the ndarray, a fast and memory-efficient array structure that allows for vectorized operations—making it much faster than using Python lists for numerical tasks. With NumPy, you can easily perform mathematical operations like squaring (np.square), square roots (np.sqrt), and even linear algebra and statistical functions. This makes it essential for tasks like scientific simulations, data processing, and machine learning.

You can learn more at the official [NumPy documentation](https://numpy.org/doc/stable/).

In [ ]:
# Time comparison 
import numpy as np

# Calculate like a list using only python
%timeit [x**2 for x in df['velocity_x']]

# Normal operation with Pandas
%timeit df['velocity_x']**2

# Using Numpy operations
%timeit np.square(df['velocity_x'])

Even though all three approaches produce the same result, the list comprehension is much slower because it runs in pure Python and processes elements one by one. Both the pandas-native (**2) and NumPy (np.square) methods are vectorized and rely on fast, compiled code. Since pandas is built on top of NumPy, it already uses NumPy under the hood—so using either approach gives you similar performance, and both are much more efficient for large datasets.

The first calculations can now be made. For example, we can calculate the distance from a point to the origin using:

$$
r = \sqrt{x^2 + y^2 + z^2}
$$

Herefore we use the library NumPy (Numerical Python). The library is used for numerical computing and offers among other things a wider range of mathematical functions. For our calculation we use `np.sqrt` to use the square root.

In [ ]:
# Import the library numpy with the shorthand np
import numpy as np

r = np.sqrt(df.loc[10, 'x']**2 + df.loc[10, 'y']**2 + df.loc[10, 'z']**2)
print(f'r =  {r}')

::: {.callout-important}
## Task 2

Calculate the absolute velocity $|\mathbf{v}| = \sqrt{v_x^2 + v_y^2 + v_z^2}$
for **every** point and store it in a new column. Check how much extra storage
this adds by comparing `df.info()` before and after.

::: {.callout-tip collapse="true"}
## Hint
Instead of a single point as in the example above, operate on the whole
columns at once (vectorised), then assign the result to a new column.
:::
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show a solution"
# Solution
df['abs_velocity'] = np.sqrt(df['velocity_x']**2 + df['velocity_y']**2 + df['velocity_z']**2)
df

### Statistical values
The function .describe() is very useful to view statistic data of a certain column. Try it out and see what data are printed out.

In [ ]:
df['abs_velocity'].describe()

### Slice the DataFrame
To further visualize the data, we cut a slice out of the three-dimensional space. The slice should represent the x-y plane. Therefore we create a new Dataframe "sliced_df" including all the data of the DataFrame "df" where the z value equals 0.

In [ ]:
# first select a z value for the slice (in this case z=0),
# then delete the rows in which z is not equal to 0
sliced_df = df[df['z'] == 0] 

# delete the z column as it no longer contains an additional values
sliced_df = sliced_df.drop('z', axis=1)

# print the sliced df
sliced_df

::: {.callout-tip}
## Self-test
How many grid points lie in the $z=0$ slice at a single timestep? Slice `sliced_df` to `Timestep == 1` and print the number of rows.
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show answer"
print(len(sliced_df[sliced_df['Timestep'] == 1.0]))

## Plotting in Python
### What is matplotlib?
Matplotlib is a versatile and widely-used library in Python for creating static, animated, and interactive visualizations. It provides a robust framework for generating a wide range of plots, from simple line charts to complex 3D graphics. With its intuitive syntax and extensive customization options, Matplotlib allows you to effectively visualize your data, helping to uncover trends, patterns, and insights. Whether you're building quick exploratory plots or publishing high-quality figures, Matplotlib serves as the foundation for turning raw data into meaningful visuals.

### Lineplot
Line plots are a fundamental visualization tool used to display data points connected by straight lines, making it easy to observe trends, patterns, or changes over a continuous variable like time. They are particularly useful for comparing multiple datasets and highlighting relationships or variations in a clear and concise manner.

To begin with, we want to plot the absolute velocity at a point over time. To do this, we must first remove all other points from the DataFrame.

::: {.callout-important}
## Task 3

Create a DataFrame containing only the point $x=0,\; y=0,\; z=0$.

::: {.callout-tip collapse="true"}
## Hint
Slice the DataFrame the same way we did before, but for both the $x$ and $y$
axes. Print the result to check it worked.
:::
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show a solution"
# Solution
oneD_df = sliced_df[(sliced_df['x'] == 0) & (sliced_df['y'] == 0)]
oneD_df = oneD_df.drop('x', axis=1)
oneD_df = oneD_df.drop('y', axis=1)
oneD_df

Now we want to visualize the data. To do this, we first import the matplolib.pyplot library. We can then plot our data using the `.plot` function. The first entry refers to the x-axis and the second to the y-axis.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(oneD_df['Timestep'], oneD_df['abs_velocity'])

A consistent definition of axis labels (and limits) is essential for creating high quality figures. The axis labels can be defined with `plt.xlabel`and `plt.ylabel` and a heading with `plt.title`. You should choose a sufficiently large font size so that they are easy to read in presentations. With `plt.xlim` limits can be set for the axes.

In [ ]:
Tmin = oneD_df['Timestep'].min()
Tmax = oneD_df['Timestep'].max()

plt.plot(oneD_df['Timestep'], oneD_df['abs_velocity'])
plt.xlabel('Timestep [-]')
plt.ylabel('Velocity [-]')
plt.title('Velocity over time')
plt.xlim(Tmin, Tmax)

Matplotlib provides extensive options to customize your line plots, allowing you to create visuals tailored to your needs. You can control aspects like the line style, color, and markers used to indicate data points.

In this example:

- `linestyle='--'` creates a dashed line.
- `color='r'` sets the line color to red.
- `marker='o'` uses circular markers for data points.
- `markersize=2` adjusts the size of the markers.
  
You can choose from a wide variety of colors (e.g., 'r' for red, 'g' for green) and marker styles (e.g., 'o' for circles, '^' for triangles).

In [ ]:
plt.plot(oneD_df['Timestep'], oneD_df['abs_velocity'],
         linestyle='--', color='r', marker='o',
         markersize=2, label='Sample Data')
plt.xlabel('Timestep [-]')
plt.ylabel('Velocity [-]')
plt.title('Velocity over time')
plt.xlim(Tmin, Tmax)

::: {.callout-note collapse="true"}
## Note
Often it doesn't make sense to make many adjustments — as you can see, the
plot above is arguably more legible *before* the extra styling. For a full
list of line styles, markers and colours, see the
[Matplotlib documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html).
:::

With the shared style applied in **Setup**, you rarely need to set fonts or sizes by hand — the stylesheet does it for you. When you *do* want a one-off override, define a variable once (e.g. `linewidth`) and reuse it, so a single change updates every call.

In [ ]:
linewidth=0.85

plt.plot(oneD_df['Timestep'], oneD_df['abs_velocity'], linewidth=linewidth)
plt.xlabel('Timestep [-]')
plt.ylabel('Velocity [-]')
plt.title('Velocity over time')
plt.xlim(Tmin, Tmax)

Several graphs can also be plotted in one plot. To do this, simply plot several functions with `plt.plot`, which are then automatically placed in a plot. To keep them apart you should add a legend. This is done with `plt.legend()`. In the brackets you can also specify the position of the legend with `loc=` (best: 0, upper right: 1, upper left: 2, lower left: 3, lower right: 4, if nothing is used it equals the 0). The different graphs are named by adding `label= name of your plot` to the plot.

::: {.callout-important}
## Task 4

Create a second DataFrame for the point $x=0,\;$ $y=$ (one grid step
further)$,\; z=0$. Then plot the velocity over time for **both** points in one
figure, with a legend.

::: {.callout-tip collapse="true"}
## Hint
Slice as before, and use `.iloc[1]` to pick the next $y$ value.
:::
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show a solution"
# Solution
# second point: same x, z but the next y grid value
oneD_df2 = df[(df['x'] == 0) & (df['y'] == df['y'].iloc[1]) & (df['z'] == 0)]

plt.plot(oneD_df['Timestep'], oneD_df['abs_velocity'],
         linewidth=linewidth, label='x=0, y=0, z=0')
plt.plot(oneD_df2['Timestep'], oneD_df2['abs_velocity'],
         linewidth=linewidth, label='x=0, y=next, z=0')
plt.xlabel('Timestep [-]')
plt.ylabel('Velocity [-]')
plt.title('Velocity over time')
plt.xlim(Tmin, Tmax)
plt.legend()

If you plot multiple graphs, there are a few more useful tricks. For example, you can create lists of the line styles and colors and then loop over them when you create your plots. This will give the different plots the desired color and linestyle. It is also practical to create a dictionary for your labels, especially for larger tasks with many variables. This way, you don't always have to rename the units and remain consistent.
At the end you can save your plots with `plt.savefig(path_data /'figures '/figure_name.png', dpi=1000)`. It is important to use meaningful names to keep the overview and to choose a high resolution.

To do this, you create a figure at the beginning in which you then plot your graph. This allows us not only to save our plots, but also to set the size of the plot beforehand. It should be noted that the values are given in inches, so pay attention to the conversion. In the example below you can see, how to make a plot with the \textwidth of the TU template, for the height we use in this case a 5:3 aspect ratio.

In [ ]:
#| label: fig-two-point-line
#| fig-cap: "Velocity magnitude over time at two neighbouring grid points."
# Create figure with an explicit size (in inches)
fig, ax = plt.subplots()

dfs = [oneD_df, oneD_df2]
var_label_dict = {"Timestep": "Timestep [-]", "abs_velocity": "Velocity [-]"}
colors = ['#006BA4', '#FF800E']
linestyles = ['-', '--']
labels = ['x = 0, y = 0, z = 0',
          f'x = 0, y = {round(df["y"].iloc[1], 5)}, z = 0']

x_var, y_var = "Timestep", "abs_velocity"
for df_line, color, ls, label in zip(dfs, colors, linestyles, labels):
    ax.plot(df_line[x_var], df_line[y_var], color=color, linestyle=ls, label=label)

ax.set_xlabel(var_label_dict.get(x_var, x_var))
ax.set_ylabel(var_label_dict.get(y_var, y_var))
ax.set_title('Velocity over time')
ax.legend()
ax.grid(True)

# To save: fig.savefig('Velocity_over_time.png')   # dpi comes from the stylesheet
plt.show()

The possibilities for designing your plots are almost unlimited. Below is a small example of how you can create a figure within a figure to zoom into the data. 

In [ ]:
'''
Step 1:
We start by plotting a normal graph. For example, take the one from your task.
It is also important to mention that you can set the size of the figure with figsize=(Width, height)(in inches).

'''

Tmin = oneD_df['Timestep'].min()
Tmax = oneD_df['Timestep'].max()

fig, axes = plt.subplots(figsize=(8,8), nrows=1, ncols=1)

plt.plot(oneD_df['Timestep'], oneD_df['abs_velocity'], label='x=0, y=0, z=0', color='#006BA4')
plt.plot(oneD_df2['Timestep'], oneD_df2['abs_velocity'],
        label=f"x=0, y={sliced_df['y'].iloc[1].round(4)}, z=0", color='#FF800E') 
plt.xlabel('Timestep [-]')
plt.ylabel('Velocity [-]')
plt.title('Velocity over time')

fig.tight_layout()

plt.legend()
plt.xlim(Tmin, Tmax)

'''
Step 2:
Now we create the layout for the zoomed-in section of the graph. To do this, we create a dictionary with all the relevant data for our layout.
This has the advantage that if we want to make adjustments, we have everything in the same place and 
if we want to look at a different part of the graph, we just need to make changes in the dictionary. 
We then create the axes and enter the sizes from the dictionary. Finally, we plot our data again in our new axes.
'''

zoom_dict = {'x': [300, 400],           # x limits for zoomed area
             'y': [1, 1.27],            # y limits for zoomed area
             'zoom_size': [0.4, 0.7,    # location in x and y (bottom left point of the zoomed window)
                           0.3, 0.3],   # relative size in x and y of zoomed window (width, height of the rectangle)
             'loc': [3, 4]              # corners to be connected to highlighting frame (step 3)
            }

# create axes for zoomed area inside the bottom plot
ax_new = axes.inset_axes(zoom_dict['zoom_size'])
ax_new.grid(False) 

# set limit of zoomed area
ax_new.set_xlim(zoom_dict['x'][0],zoom_dict['x'][1])
ax_new.set_ylim(zoom_dict['y'][0],zoom_dict['y'][1])

# hide ticks --> not needed since a frame is used to highlight zoomed area
ax_new.set_xticks([])
ax_new.set_yticks([])

# also plot data to new axis
ax_new.plot(oneD_df['Timestep'], oneD_df['abs_velocity'],
            color='#006BA4', linestyle='-', linewidth=0.85)
ax_new.plot(oneD_df2['Timestep'], oneD_df2['abs_velocity'],
            color='#FF800E', linestyle='-', linewidth=0.85)

'''
Step 3:
At the end we can create a frame that points to the zoomed area in the full scale view.
After that, we can already save our beautiful plot.
'''

from mpl_toolkits.axes_grid1.inset_locator import mark_inset

# loc1 and loc2 are the corners of the zoomed plot to be connected with frame
# Corners are labeled as:
#       2 - 1
#       |.  |
#       3 - 4
mark_inset(axes, ax_new, loc1=zoom_dict['loc'][0], loc2=zoom_dict['loc'][1], 
           #fc="none", 
           fc="C0", # fill color of highlighting frame
           ec="C1", # edge color of highlighting frame
           alpha=0.5)

# plt.savefig('velocity_over_time.pdf')

::: {.callout-important}
## Task 5

Create a properly formatted plot (axis labels, title, …) of the **mean**
velocity over time, and save it as a PNG or PDF. Submit it in **Crowdmark**.

::: {.callout-tip collapse="true"}
## Hint
Use `.groupby('Timestep')` to average over all points at each timestep.
:::
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show a solution"
# Possible solution

# Group by timestep and average the velocity magnitude at each timestep
mean_velocity = df.groupby('Timestep')['abs_velocity'].mean()

fig, ax = plt.subplots()
ax.plot(mean_velocity.index, mean_velocity.values)
ax.set_xlabel('Timestep [-]')
ax.set_ylabel('Mean velocity [-]')
ax.set_title('Mean velocity over time')
ax.grid(True)

# To save: fig.savefig('mean_velocity.png')

### Tricontourplot

Tricontour plots are used to visualize scalar fields over irregularly spaced triangular grids, making them ideal for representing data distributed over non-uniform domains. They display contour lines or filled contours, helping to reveal variations and gradients in the scalar field effectively.

Plot the absolute velocity using the tricontourf function. The third entry specifies which size is colored.

#### Create a DataFrame including only the first Timestep
Since we only want to look at one time step for the time being, we can remove the other time steps from the DataFrame to avoid transporting the data unnecessarily.

In [ ]:
# Slice the DataFrame only including the first Timestep
t1_df = sliced_df[sliced_df['Timestep'] == 1.0] 

# Drop the column z
t1_df = t1_df.drop('Timestep', axis=1)

# print the sliced df
t1_df

In [ ]:
plt.tricontourf(t1_df['x'], t1_df['y'], t1_df['abs_velocity'])
plt.title("TS 1")

To be able to evaluate the plot, we need to add a colorbar to assign the different color in values. With `cmap='colormap name'` you can specify which colormap should be used. [Here](https://matplotlib.org/stable/users/explain/colors/colormaps.html) you can find a selection of different colormaps. You can also set levels, this defines how many contours (or at what values) to draw between your vmin and vmax. 

In [ ]:
#| label: fig-contour-ts1
#| fig-cap: "Filled contour of velocity magnitude on the z = 0 plane at timestep 1."
plt.tricontourf(t1_df['x'], t1_df['y'], t1_df['abs_velocity'], cmap = 'viridis', levels=500)
cbar = plt.colorbar() 
cbar.set_label('Absolute Velocity') 
plt.title("TS 1")
plt.show()

::: {.callout-tip}
## Self-test
Redraw the filled contour for timestep 1, but colour by the $x$-velocity `velocity_x` instead of the magnitude. Notice that it now takes both signs.
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show answer"
plt.tricontourf(t1_df['x'], t1_df['y'], t1_df['velocity_x'],
                cmap='viridis', levels=200)
plt.colorbar(label='velocity_x')
plt.title('TS 1 - x-velocity')
plt.show()

### Subplots
Subplots are a feature in data visualization that allows multiple plots to be displayed within a single figure, arranged in a grid-like layout. They are ideal for comparing different datasets or aspects of data side-by-side, enabling a clearer and more comprehensive analysis.

As we have already learned in the section we first create our own figure. A figure is an object that contains all the plot elements. This includes not only the size, but also the number of plots. It can contain many axes, `nrows` as the amount of rows and `ncols`defines the ammount of columns. With `tight_layout=True` it automatically adjusts subplots parametes so that the subplots fits in the figure area. With `ax[row,col]` you can determine the position of the individual plots.

In [ ]:
fig, ax = plt.subplots(figsize=(10,10), nrows=2, ncols=2, tight_layout=True)

levels = 500
filtered_df = df[df['Timestep'] == 5]
# the tricontour plot is defined as "im" short for "image",
# so it can be used to create a colorbar that is assigned to the correct subplot
im = ax[0,0].tricontourf(filtered_df['x'], filtered_df['y'],
                         filtered_df['abs_velocity'], cmap = 'viridis', levels=levels)
cbar = fig.colorbar(im, pad = 0.01)
ax[0,0].set_title('TS 5')

filtered_df = df[df['Timestep'] == 10]
im = ax[0,1].tricontourf(filtered_df['x'], filtered_df['y'],
                         filtered_df['abs_velocity'], cmap = 'viridis', levels=levels)
cbar = fig.colorbar(im, pad = 0.01)
ax[0,1].set_title('TS 10')

filtered_df = df[df['Timestep'] == 15]
im = ax[1,0].tricontourf(filtered_df['x'], filtered_df['y'],
                         filtered_df['abs_velocity'], cmap = 'viridis', levels=levels)
cbar = fig.colorbar(im, pad = 0.01)
ax[1,0].set_title('TS 15')

filtered_df = df[df['Timestep'] == 20]
im = ax[1,1].tricontourf(filtered_df['x'], filtered_df['y'],
                         filtered_df['abs_velocity'], cmap = 'viridis', levels=levels)
cbar = fig.colorbar(im, pad = 0.01)
ax[1,1].set_title('TS 20')

Now we have plotted the velocity profiles of different time steps side by side. However, we often have the same lines of code with only slight differences and with small changes to what we want to plot, we would have to change them everywhere. For this reason, it usually makes sense to use loops to create the plots, especially with subplots.

In [ ]:
fig, ax = plt.subplots(figsize=(10,10), nrows=2, ncols=2, tight_layout=True)

for i in range(4):
    filtered_df = df[df['Timestep'] == (i + 1) * 5]
    im = ax[divmod(i, 2)].tricontourf(filtered_df['x'], filtered_df['y'],
                                 filtered_df['abs_velocity'], cmap = 'viridis', levels=levels) # divmod display the quotient and the remainder 
    cbar = fig.colorbar(im, pad = 0.01)
    ax[divmod(i, 2)].set_title(f'TS {(i + 1) * 5}')

If we now want to look at other time steps, for example, we can simply copy the code from above and only have to replace a single thing.

::: {.callout-important}
## Task 6

Adjust the subplot code above so that timesteps **2, 4, 6 and 8** are
displayed. Save the figure for submission in **Crowdmark**.
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show a solution"
# Solution
fig, ax = plt.subplots(figsize=(10,10), nrows=2, ncols=2, tight_layout=True)

for i in range(4):
    filtered_df = df[df['Timestep'] == (i + 1) * 2]
    im = ax[divmod(i, 2)].tricontourf(filtered_df['x'], filtered_df['y'],
                                 filtered_df['abs_velocity'], cmap = 'viridis', levels=levels) # divmod display the quotient and the remainder 
    cbar = fig.colorbar(im, pad = 0.01)
    ax[divmod(i, 2)].set_title(f'TS {(i + 1) * 2}')

# plt.savefig('task6.png')

For subplots with colorbars, instead of having a colorbar for each individual plot, it makes sense to create one that applies to all plots. This makes the plots easier to compare, as the scaling is the same everywhere. We can also make our code even more variable. If we define the time steps and the distance between the time steps beforehand, we only have to change the number of time steps, for example, and instead of 4 we can create 8 subplots.

In [ ]:
#| label: fig-subplot-grid
#| fig-cap: "Velocity-magnitude fields across eight timesteps with a shared colour bar."
# Set the number of timesteps to visualize
num_timesteps = 8  # You can change this to plot a different number of timesteps
timestep_interval = 5  # Interval between each selected timestep
ncols = 2  # Number of columns for subplot layout
cmap = 'viridis'  # Color map to use for the plots
levels = 500  # Number of contour levels
labelsize = 14  # Font size for axis tick labels

# Create a list of specific timesteps we want to plot, spaced by timestep_interval
timestep_values = [(i + 1) * timestep_interval for i in range(num_timesteps)]

# Filter the DataFrame to only include the selected timesteps
filtered_df = df[df["Timestep"].isin(timestep_values)]

# Aggregate the minimum and maximum velocity magnitudes for each selected timestep
agg_results = filtered_df.groupby("Timestep")["abs_velocity"].agg(["min", "max"])

# Determine the global minimum and maximum velocity values for consistent color scaling
vmin, vmax = agg_results["min"].min(), agg_results["max"].max()

# Calculate the number of rows needed based on the number of columns
nrows = -(-num_timesteps // ncols)  # Ceiling division to ensure enough rows

# Create a figure and array of axes for the subplots
fig, ax = plt.subplots(
    nrows, ncols,
    figsize=(ncols * 5, nrows * 5),  # Size of the full figure
    constrained_layout=True,  # Automatically adjust spacing to fit everything
    sharex=True, sharey=True  # Share x and y axes across subplots
)
ax = ax.flatten()  # Flatten the axes array for easy indexing

# Loop through each timestep and create a contour plot
for i, timestep in enumerate(timestep_values):
    # Filter data for the current timestep
    timestep_df = filtered_df[filtered_df["Timestep"] == timestep]
    
    # Create a filled contour plot for the current timestep
    im = ax[i].tricontourf(
        timestep_df["x"], timestep_df["y"], timestep_df["abs_velocity"],
        vmin=vmin, vmax=vmax, cmap=cmap, levels=levels
    )
    
    # Set the title and customize tick labels
    ax[i].set_title(f"Timestep {timestep}")
    ax[i].tick_params(direction="in", labelsize=labelsize)

# Remove any unused axes if the number of timesteps is less than the total subplots
for i in range(num_timesteps, len(ax)):
    fig.delaxes(ax[i])

# Add a horizontal colorbar that spans the width of the subplots
cbar = fig.colorbar(im, ax=ax[:num_timesteps], orientation="horizontal", fraction=0.5, pad=0.01)
cbar.set_label("Velocity Magnitude")
cbar.ax.tick_params(labelsize=labelsize)

# Display the final figure
plt.show()

# fig.savefig('velocity_profiles.png')


We have now written our code quite flexibly so that we can use it more often. However, to keep our notebook simple and clear, we can write our code as a function. To do this, we basically just have to define a name with the parameters and insert our code from above.

In [ ]:
import matplotlib.pyplot as plt

def plot_velocity_profiles(
    df, 
    num_timesteps=8, 
    timestep_interval=5, 
    ncols=2, 
    cmap='viridis', 
    levels=500, 
    labelsize=14,
    save_path=None
):
    """
    Plot multiple velocity profiles from a DataFrame across selected timesteps.
    
    Parameters:
    - df: DataFrame with 'Timestep', 'x', 'y', 'abs_velocity' columns
    - num_timesteps: Number of timesteps to plot
    - timestep_interval: Interval between timesteps
    - ncols: Number of subplot columns
    - cmap: Colormap for the velocity field
    - levels: Number of contour levels
    - fontsize: Font size for titles
    - labelsize: Font size for ticks
    - save_path: Optional path to save the figure (Path object or string)
    """
    
    # Select the timesteps
    timestep_values = [(i + 1) * timestep_interval for i in range(num_timesteps)]
    filtered_df = df[df["Timestep"].isin(timestep_values)]
    
    # Determine vmin and vmax globally across selected timesteps
    agg_results = filtered_df.groupby("Timestep")["abs_velocity"].agg(["min", "max"])
    vmin, vmax = agg_results["min"].min(), agg_results["max"].max()
    
    # Setup figure and axes
    nrows = -(-num_timesteps // ncols)  # Ceiling division
    fig, ax = plt.subplots(
        nrows, ncols, figsize=(ncols * 5, nrows * 5),
        constrained_layout=True, sharex=True, sharey=True
    )
    ax = ax.flatten()
    
    # Plot each timestep
    for i, timestep in enumerate(timestep_values):
        timestep_df = filtered_df[filtered_df["Timestep"] == timestep]
        im = ax[i].tricontourf(
            timestep_df["x"], timestep_df["y"], timestep_df["abs_velocity"],
            vmin=vmin, vmax=vmax, cmap=cmap, levels=levels
        )
        ax[i].set_title(f"Timestep {timestep}")
        ax[i].tick_params(direction="in", labelsize=labelsize)
    
    # Hide any unused subplots
    for empty_ax in ax[num_timesteps:]:
        empty_ax.remove()
    
    # Add a horizontal colorbar
    cbar = fig.colorbar(im, ax=ax[:num_timesteps], orientation="horizontal", fraction=0.5, pad=0.01)
    cbar.set_label("Velocity Magnitude")
    cbar.ax.tick_params(labelsize=labelsize)
    
    # Save the figure if a path is provided
    if save_path:
        fig.savefig(save_path, dpi=1000)
    
    plt.show()

In [ ]:
plot_velocity_profiles(df, num_timesteps=6, timestep_interval=10, ncols=3)

As you can see, we only need one line of code and can set the number of plots, as well as the time interval, font size etc. as required.

As you can see, I have made a few more changes. For example, that the axis labels are only on the outer plots or that the ticks on the axes point inwards. You can change a lot of different things when designing your graphs. Feel free to play around and find your preferences.

Hint: This could be useful for a possible plotting contest :-O

### Scatter and Hexbin plots
It is not only possible to plot in spatial space, but sometimes data evaluation is also more useful in state space, e.g. to obtain a dependency on a certain variable.
In these plots a huge number of sample points are shown to display some kind of distribution.

To visualize this data, multiple kind of plots exist, each having their distinct advantages. In the following we will show some of our personal favorites:
- scatter plots: A scatter plot displays individual data points as dots on a 2D plane, showing the relationship between two variables.)
- hexbin plots: A hexbin plot aggregates data points into hexagonal bins, where the color intensity represents the density of points in each bin, making it useful for visualizing large datasets.)

In [ ]:
#| label: fig-scatter-hexbin
#| fig-cap: "Scatter vs. hexbin of the velocity components: hexbin shows the density a scatter plot hides."
# Create figure and gridspec
fig = plt.figure(figsize=(10, 6))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.1])

# Create subplots using gridspec
ax = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
cbar_ax = fig.add_subplot(gs[0, 2])

# Scatter plot
ax[0].scatter(t1_df["velocity_x"], t1_df["velocity_y"], s=0.5, color="C0")
ax[0].set_title("scatter")

# Hexbin plot
im = ax[1].hexbin(t1_df["velocity_x"], t1_df["velocity_y"], cmap='Blues', mincnt=1)
ax[1].set_title("hexbin")

# Labels
ax[0].set_xlabel('$v_x$ [-]')
ax[1].set_xlabel('$v_x$ [-]')
ax[0].set_ylabel('$v_y$ [-]')

# Colorbar for hexbin plot
fig.colorbar(im, cax=cbar_ax)
plt.show()

In the plot above we see the benefit of the hexbin over the scatter plot for data distribution. While the first scatter plot (left) is able to visualize the area in which the data points are located, we lose lots of details of the actual distribution of the data due to overlapping points. We can slightly overcome this problem, by using a transparent marker (middle). However, in the areas with a high point density we still lose lot’s of insights. Here, the hexbin gives the best results, since all the data points inside the hexbin is summed up and we can visualize the data-distribution (and we even can choose the color map for this).

::: {.callout-tip}
## Self-test
Make a hexbin of `velocity_x` against `velocity_z` for the first timestep. Is the joint distribution as round as the $v_x$-$v_y$ one?
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show answer"
plt.hexbin(t1_df['velocity_x'], t1_df['velocity_z'], cmap='Blues', mincnt=1)
plt.xlabel(r'$v_x$ [-]')
plt.ylabel(r'$v_z$ [-]')
plt.colorbar(label='count')
plt.show()

### How to define a plotting style
One of the key aspects in the generation of high-quality figures is a consistent design of the plots, which includes
- Font-size
- Typeface
- Text-rendering
- Color-schemes
- Axis position and design
- ...

If you have to do a lot of plotting, it can be a tedious and time-consuming job. In the following, we will show you an easy way to define a consistent design using **matplotlib stylesheets**.

The Pros of this methods are:
- Central style definition
- Easy reuse
- Highly flexible
- Very clean code

After answering the question Why to use matplotlib stylesheets, we will now give you a short overview on how to use matplotlib plotting styles. A broader overview can also be found in [here](https://matplotlib.org/stable/users/explain/customizing.html).

There are several [stylesheets](https://matplotlib.org/stable/gallery/style_sheets/style_sheets_reference.html) on the matplotlib page that can be used or modified. Next, we will create our own stylesheet using the things we have learned so far.

::: {.callout-important}
## Task 7

Experiment with different marker sizes, line widths and colours, then set your
favourites in the stylesheet defined below. Because roughly 8% of men and 0.4%
of women have a colour-vision deficiency, avoid relying on red-vs-green
contrasts. Browse colour palettes
[here](https://jrnold.github.io/ggthemes/reference/tableau_color_pal.html).
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show a solution"
# Possible solution
from pathlib import Path

style_path = Path("myStyle.mplstyle")

style_lines = [
    "# Fonts / math (mathtext -> no external LaTeX needed)",
    "font.family      : serif",
    "mathtext.fontset : cm",
    "font.size        : 12",
    "",
    "# Lines",
    "lines.markersize : 3",
    "lines.linewidth  : 1.0",
    "",
    "# Figure",
    "figure.figsize   : 7.0, 4.2   # inches (5:3 aspect ratio)",
    "savefig.dpi      : 300",
    "",
    "# Ticks",
    "xtick.direction  : in",
    "ytick.direction  : in",
    "",
    "# Legend",
    "legend.frameon   : false",
    "legend.fontsize  : medium",
    "",
    "# Colour-blind-friendly colour cycle (tableau 10-blind)",
    "axes.prop_cycle  : cycler('color', ['006BA4', 'FF800E', 'ABABAB', '595959', '5F9ED1', 'C85200', '898989', 'A2C8EC', 'FFBC79', 'CFCFCF'])",
    "patch.facecolor  : 006BA4",
    "",
    "image.cmap       : viridis",
]

style_path.write_text("\n".join(style_lines))
print(f"Custom style saved to: {style_path}")

::: {.callout-important}
## Task 8

Now test your stylesheet: create a line plot with at least two curves
**without** specifying colours or font sizes by hand. Load your stylesheet with
`plt.style.use(['path/to/style.mplstyle'])`, save the figure, and submit it in
**Crowdmark**.
:::

In [ ]:
#| code-fold: true
#| code-summary: "Show a solution"
# Possible solution

# Use the stylesheet we just created
plt.style.use(['myStyle.mplstyle'])

dfs = [oneD_df, oneD_df2]
labels = ['x = 0, y = 0, z = 0',
          f'x = 0, y = {round(df["y"].iloc[1], 5)}, z = 0']

for df_line, label in zip(dfs, labels):
    plt.plot(df_line['Timestep'], df_line['abs_velocity'], label=label)

plt.xlabel('Timestep [-]')
plt.ylabel('Velocity [-]')
plt.title('Velocity over time')
plt.legend()

# To save: plt.savefig('styled_lineplot.png')

As we can see, we can get a plot with a certain design without much effort. We will continue to use the stylesheet we have created in future tutorials. Feel free to customize or extend the stylesheet as the tutorial progresses. You can make adjustments in the file itself, further information can be found in the [documentation](https://matplotlib.org/stable/users/explain/customizing.html).


## Summary
The brave ones who have held out to the end are now able to:

- Read and handle data in tabular form:
    - Explore data structure and summarize information
    - Add, remove, and manipulate data
    - Perform calculations on the data
- Visualize data through:
    - Line plots
    - Scatter and hexbin plots
    - Tricontour plots
    - Subplots for comparison
- Customize plotting style to suit your needs

### Guidelines
Here are a few guidelines you can follow for the plots in the subsequent tutorials.
- **General plotting style**
  - Consistent style
    - colorscheme, axis labels, ...
- **Figure quality and resolution** 
  - `.pdf`: Line-plots and small plots
  - `.png/.jpeg`: Plots with color maps / contour plots
  - Sufficient resolution (~1000 dpi)
- **Labels and units**
  - Consistent use
  - Units non-italic
  - Typical formatting 
    - *T* / K -> recommended by [SI guidelines](https://www.nist.gov/pml/special-publication-811/nist-guide-si-chapter-7-rules-and-style-conventions-expressing-values)
    - *T* in K
    - *T* (K)
    - *T* [K]